# MLflow Run Inventory & Pruning Helper

It supports two situations:

1. **Full tracking store**: you have both `mlflow.db` and an MLflow artifact store.  
2. **Artifact-only export**: you only have per-run `artifacts/` folders (common if you zipped `mlruns/` but not `mlflow.db`).

For each run, the notebook reports:
- last modified time (as a proxy for "age" when DB timestamps are unavailable),
- whether it looks complete (timings + baseline metrics + FACTER metrics),
- whether it contains stage error logs,
- artifact size and key artifact folders.

It then produces:
- `runs_inventory.csv` (all runs),
- `runs_candidates.csv` (runs you likely want to keep),
- `keep_run_ids.txt` (one run_id per line).

Nothing is deleted. Optionally, you can **copy** selected runs into a `pruned_mlruns/` folder for zipping.


In [61]:
from __future__ import annotations

import os
import json
import zipfile
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_upwards(target_rel: Path) -> Path | None:
    """Search upwards from CWD for a relative path that exists."""
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        cand = (base / target_rel).resolve()
        if cand.exists():
            return cand
    return None


plt.rcParams["figure.figsize"] = (10, 5)

# CONFIG

ZIP_PATH = find_upwards(Path("mlruns.zip")) or Path("mlruns.zip")
BASE_DIR = ZIP_PATH.resolve().parent if ZIP_PATH.exists() else Path.cwd().resolve()

EXTRACT_DIR = BASE_DIR / "mlruns/extracted"

DB_PATH_HINT = find_upwards("mlflow.db")  # e.g., Path("mlflow.db")

# Selection defaults
DAYS_BACK = 5                    # "recent" window
REQUIRE_FINISHED = True          # only possible if DB exists; otherwise uses artifact heuristics
REQUIRE_NO_STAGE_ERRORS = True
REQUIRE_TIMINGS = True

# If you want to keep only runs that completed the online loop, set True:
REQUIRE_FACTER_METRICS = True

OUTPUT_DIR = BASE_DIR / "mlruns/output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Config:")
print("  ZIP_PATH:", ZIP_PATH.resolve() if ZIP_PATH.exists() else ZIP_PATH)
print("  BASE_DIR:", BASE_DIR)
print("  EXTRACT_DIR:", EXTRACT_DIR)
print("  OUTPUT_DIR:", OUTPUT_DIR)
print("  DAYS_BACK:", DAYS_BACK)
print("  REQUIRE_FACTER_METRICS:", REQUIRE_FACTER_METRICS)


Config:
  ZIP_PATH: C:\Users\daimy\Uni\MasterAI\FACT\facter-repr\scripts\mlruns.zip
  BASE_DIR: C:\Users\daimy\Uni\MasterAI\FACT\facter-repr\scripts
  EXTRACT_DIR: C:\Users\daimy\Uni\MasterAI\FACT\facter-repr\scripts\mlruns\extracted
  OUTPUT_DIR: C:\Users\daimy\Uni\MasterAI\FACT\facter-repr\scripts\mlruns\output
  DAYS_BACK: 5
  REQUIRE_FACTER_METRICS: True


### Extract zip

In [62]:
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

if ZIP_PATH.exists():
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(EXTRACT_DIR)
    print("Extracted to:", EXTRACT_DIR.resolve())
else:
    raise FileNotFoundError(f"ZIP_PATH not found: {ZIP_PATH.resolve()}")


Extracted to: C:\Users\daimy\Uni\MasterAI\FACT\facter-repr\scripts\mlruns\extracted


### Locate mlruns

In [63]:
def find_first(root: Path, name: str) -> Path | None:
    matches = list(root.rglob(name))
    return sorted(matches, key=lambda p: len(str(p)))[0] if matches else None

mlruns_dir = find_first(EXTRACT_DIR, "mlruns")
db_path = DB_PATH_HINT if DB_PATH_HINT else find_first(EXTRACT_DIR, "mlflow.db")

print("mlruns_dir:", mlruns_dir)
print("db_path:", db_path)

if mlruns_dir is None or not mlruns_dir.exists():
    raise FileNotFoundError("Could not find an 'mlruns' directory inside the extracted data.")


mlruns_dir: C:\Users\daimy\Uni\MasterAI\FACT\facter-repr\scripts\mlruns\extracted\mlruns
db_path: C:\Users\daimy\Uni\MasterAI\FACT\facter-repr\mlflow.db


### Helpers + artifact-only scan

In [64]:
RUN_STATUS = {1:"SCHEDULED",2:"RUNNING",3:"FINISHED",4:"FAILED",5:"KILLED"}

def dir_size_mb(p: Path) -> float:
    total = 0
    for root, _, files in os.walk(p):
        for fn in files:
            fp = Path(root) / fn
            try:
                total += fp.stat().st_size
            except FileNotFoundError:
                pass
    return total / (1024 * 1024)

def newest_mtime(p: Path) -> datetime | None:
    mt = 0.0
    for root, _, files in os.walk(p):
        for fn in files:
            fp = Path(root) / fn
            try:
                mt = max(mt, fp.stat().st_mtime)
            except FileNotFoundError:
                pass
    return datetime.fromtimestamp(mt) if mt > 0 else None

def safe_json_load(p: Path):
    try:
        return json.loads(p.read_text(encoding="utf-8"))
    except Exception:
        return None

def scan_artifact_run(run_dir: Path) -> dict:
    run_id = run_dir.name
    art_dir = run_dir / "artifacts"

    subdirs = sorted([d.name for d in art_dir.iterdir() if d.is_dir()]) if art_dir.exists() else []
    results_dir = art_dir / "results"

    timings_path = results_dir / "timings.json"
    has_timings = timings_path.exists()
    timings = safe_json_load(timings_path) if has_timings else None

    baseline_metrics = list(results_dir.glob("baseline_metrics_*.json")) if results_dir.exists() else []
    facter_metrics = list(results_dir.glob("facter_metrics_*.json")) if results_dir.exists() else []

    stage_errors = list((art_dir / "stage_errors").glob("*.txt")) if (art_dir / "stage_errors").exists() else []

    total_seconds = None
    if isinstance(timings, dict):
        total_seconds = timings.get("stage.TOTAL.seconds") or timings.get("TOTAL.seconds") or timings.get("TOTAL")

    return {
        "run_id": run_id,
        "has_artifacts": art_dir.exists(),
        "artifact_subdirs": ",".join(subdirs),
        "has_results_dir": results_dir.exists(),
        "has_timings": has_timings,
        "total_seconds": total_seconds,
        "n_baseline_metrics_files": len(baseline_metrics),
        "n_facter_metrics_files": len(facter_metrics),
        "has_stage_errors": len(stage_errors) > 0,
        "n_stage_errors": len(stage_errors),
        "artifact_size_mb": dir_size_mb(art_dir) if art_dir.exists() else 0.0,
        "last_modified": newest_mtime(run_dir),
    }

def scan_mlruns_artifact_only(mlruns_dir: Path) -> pd.DataFrame:
    # Two common layouts:
    # (A) standard file-store: mlruns/<exp_id>/<run_id>/...
    # (B) artifact-only: mlruns/<run_id>/artifacts/...
    rows = []

    # Detect layout B
    top = [d for d in mlruns_dir.iterdir() if d.is_dir() and (d / "artifacts").exists()]
    if top:
        for rd in top:
            rows.append(scan_artifact_run(rd))
        return pd.DataFrame(rows)

    # Layout A
    exp_dirs = [d for d in mlruns_dir.iterdir() if d.is_dir()]
    for exp in exp_dirs:
        run_dirs = [d for d in exp.iterdir() if d.is_dir() and (d / "artifacts").exists()]
        for rd in run_dirs:
            rec = scan_artifact_run(rd)
            rec["experiment_dir"] = exp.name
            rows.append(rec)

    return pd.DataFrame(rows)

runs_art = scan_mlruns_artifact_only(mlruns_dir)
print("Runs discovered:", len(runs_art))
display(runs_art.head(10))


Runs discovered: 64


,run_id,has_artifacts,artifact_subdirs,has_results_dir,has_timings,total_seconds,n_baseline_metrics_files,n_facter_metrics_files,has_stage_errors,n_stage_errors,artifact_size_mb,last_modified
0,0266519be494477c8bac7b275444342c,True,"config,data,results",True,True,None,2,1,False,0,40.018242,2026-01-30 17:50:22.803243
1,0d93b949d49b450a9810d0c6cca572df,True,"config,data,results",True,False,None,2,0,False,0,16.501107,2026-01-30 17:50:19.772477
2,10278628076d4d58b76ef0f39f9e2f64,True,"config,stage_errors",False,False,None,0,0,True,1,0.000138,2026-01-30 17:50:20.548159
3,12bca8b8ff5b4d1f9de75366958627bc,True,config,False,False,None,0,0,False,0,0.000085,2026-01-30 17:50:22.002570
4,15bfb011d668420ca2064d2a7ac8b111,True,"config,data,results",True,False,None,2,0,False,0,3.634854,2026-01-30 17:50:19.513934
5,1783da60d1c24a45a3277014db40cc3f,True,config,False,False,None,0,0,False,0,0.000085,2026-01-30 17:50:21.436331
6,1b39d684c32b4df2ac523e64f4099d0b,True,"config,stage_errors",False,False,None,0,0,True,1,0.000674,2026-01-30 17:50:22.613876
7,1b6537d0429a4e3a88f3de099c264ee7,True,"config,data,results,stage_errors",True,False,None,2,0,True,1,23.559511,2026-01-30 17:50:20.543569
8,1dea76f57e284aa19888bbeb16402c17,True,"config,data",False,False,None,0,0,False,0,2.115558,2026-01-30 17:50:21.580544
9,1e971b98c37f4222ac8803c8a21a969b,True,"config,data,results",True,True,None,2,1,False,0,23.865140,2026-01-30 17:50:22.414034


### If we have a db path, load run status/timestamps from sqlite

In [65]:
def load_runs_from_sqlite(db_path: Path) -> pd.DataFrame:
    import sqlite3
    conn = sqlite3.connect(str(db_path))
    cur = conn.cursor()

    cur.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = {r[0] for r in cur.fetchall()}
    if "runs" not in tables:
        conn.close()
        raise RuntimeError("sqlite DB does not contain a 'runs' table. Cannot load statuses.")

    cols = [r[1] for r in cur.execute("PRAGMA table_info(runs);").fetchall()]
    want = ["run_uuid", "experiment_id", "status", "start_time", "end_time", "artifact_uri"]
    avail = [c for c in want if c in cols]

    q = f"SELECT {', '.join(avail)} FROM runs"
    cur.execute(q)
    rows = cur.fetchall()
    conn.close()

    df = pd.DataFrame(rows, columns=avail)

    # Convert timestamps (ms since epoch) if numeric
    for c in ["start_time", "end_time"]:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], unit="ms", errors="coerce")

    # Normalize status to string robustly
    if "status" in df.columns:
        def to_status_str(x):
            if x is None or (isinstance(x, float) and pd.isna(x)):
                return ""
            # If already a string like "FAILED"/"FINISHED"
            if isinstance(x, str):
                return x.strip().upper()
            # If numeric code
            try:
                xi = int(x)
                return RUN_STATUS.get(xi, str(xi))
            except Exception:
                return str(x)

        df["status_str"] = df["status"].apply(to_status_str)

    return df

db_df = None
if db_path is not None and Path(db_path).exists():
    try:
        db_df = load_runs_from_sqlite(Path(db_path))
        print("Loaded DB runs:", len(db_df))
        display(db_df.head(10))
    except Exception as e:
        print("DB load failed:", e)
else:
    print("No mlflow.db found in the extracted data. Proceeding with artifact-only analysis.")


Loaded DB runs: 51


,run_uuid,experiment_id,status,start_time,end_time,artifact_uri,status_str
0,ac9f2309efc449c387db350945057faa,1,FAILED,2026-01-24 09:58:22.306,2026-01-24 11:38:45.599,file:///gpfs/home4/scur0063/facter-repr/mlruns...,FAILED
1,e0f862f33a844cd89122f41bfafbd63f,1,FAILED,2026-01-24 11:39:28.558,2026-01-24 15:25:10.395,file:///gpfs/home4/scur0063/facter-repr/mlruns...,FAILED
2,acab8765e67e4245892ade925d3cc9ec,1,FAILED,2026-01-24 15:27:53.132,2026-01-24 15:30:57.412,file:///gpfs/home4/scur0063/facter-repr/mlruns...,FAILED
3,3d66a67e821f46bcb4c6d6cfa2feb848,1,FAILED,2026-01-24 15:31:34.199,2026-01-24 15:41:32.618,file:///gpfs/home4/scur0063/facter-repr/mlruns...,FAILED
4,987a89dc8f2a418db897c23db401f2a4,1,FINISHED,2026-01-24 15:43:33.924,2026-01-24 15:53:37.686,file:///gpfs/home4/scur0063/facter-repr/mlruns...,FINISHED
5,aecd25030fd64159a41ab5a589d5baad,1,FINISHED,2026-01-25 09:22:47.152,2026-01-25 09:25:58.731,file:///gpfs/home4/scur0063/facter-repr/mlruns...,FINISHED
6,12bca8b8ff5b4d1f9de75366958627bc,1,FAILED,2026-01-25 09:25:59.171,2026-01-25 09:26:56.499,file:///gpfs/home4/scur0063/facter-repr/mlruns...,FAILED
7,8d7c0b405eca491384b323b45f35682e,1,FINISHED,2026-01-25 09:30:43.320,2026-01-25 09:33:46.735,file:///gpfs/home4/scur0063/facter-repr/mlruns...,FINISHED
8,927ce9aa8baa44f0ac35ff0a85f52214,1,FAILED,2026-01-25 10:05:06.457,2026-01-25 10:05:26.916,file:///gpfs/home4/scur0063/facter-repr/mlruns...,FAILED
9,1783da60d1c24a45a3277014db40cc3f,1,FAILED,2026-01-25 10:06:21.817,2026-01-25 10:06:26.646,file:///gpfs/home4/scur0063/facter-repr/mlruns...,FAILED


### Merge DB info (if present) + artifacts info

In [66]:
df = runs_art.copy()

if db_df is not None and not db_df.empty:
    df = df.merge(
        db_df.rename(columns={"run_uuid": "run_id"}),
        on="run_id",
        how="left",
        suffixes=("", "_db"),
    )
    df["time_ref"] = df["end_time"].fillna(df["start_time"]).fillna(df["last_modified"])
else:
    df["time_ref"] = df["last_modified"]

df["days_ago"] = (pd.Timestamp.now() - pd.to_datetime(df["time_ref"])).dt.total_seconds() / 86400.0

df["complete_baseline"] = (df["n_baseline_metrics_files"] > 0) & df["has_timings"] & (~df["has_stage_errors"])
df["complete_facter"] = (df["n_facter_metrics_files"] > 0) & df["has_timings"] & (~df["has_stage_errors"])

df["score"] = (
    2.0 * df["has_timings"].astype(float)
    + 2.0 * (df["n_baseline_metrics_files"] > 0).astype(float)
    + 2.0 * (df["n_facter_metrics_files"] > 0).astype(float)
    + 1.0 * df["has_results_dir"].astype(float)
    - 3.0 * df["has_stage_errors"].astype(float)
)

df = df.sort_values(["score", "time_ref"], ascending=[False, False]).reset_index(drop=True)

print("Inventory rows:", len(df))
display(df.head(20))


Inventory rows: 64


,run_id,has_artifacts,artifact_subdirs,has_results_dir,has_timings,total_seconds,n_baseline_metrics_files,n_facter_metrics_files,has_stage_errors,n_stage_errors,...,status,start_time,end_time,artifact_uri,status_str,time_ref,days_ago,complete_baseline,complete_facter,score
0,f3a63f513dba4596bacf85f38387c0d0,True,"config,data,results",True,True,None,2,1,False,0,...,NaN,NaT,NaT,NaN,NaN,2026-01-30 17:50:21.497680,0.000023,True,True,7.0
1,85f221e220a14541b8a2916126eb4ad0,True,"config,data,results",True,True,None,2,1,False,0,...,NaN,NaT,NaT,NaN,NaN,2026-01-30 17:50:19.789174,0.000042,True,True,7.0
2,0266519be494477c8bac7b275444342c,True,"config,data,results",True,True,None,2,1,False,0,...,FINISHED,2026-01-28 15:26:41.243,2026-01-28 17:49:33.818,file:///gpfs/home4/scur0063/facter-repr/mlruns...,FINISHED,2026-01-28 17:49:33.818000,2.000575,True,True,7.0
3,927cf28d77a94d95a9ac828e2fc8b730,True,"config,data,results",True,True,None,2,1,False,0,...,FINISHED,2026-01-28 12:18:28.048,2026-01-28 15:43:04.184,file:///gpfs/home4/scur0063/facter-repr/mlruns...,FINISHED,2026-01-28 15:43:04.184000,2.088418,True,True,7.0
4,88667f8b7aaa4ee4aacca2a19911a32b,True,"config,data,results",True,True,None,2,1,False,0,...,FINISHED,2026-01-28 12:43:57.560,2026-01-28 12:59:42.209,file:///gpfs/home4/scur0063/facter-repr/mlruns...,FINISHED,2026-01-28 12:59:42.209000,2.201866,True,True,7.0
5,b11fa34edc0946f0b382275066f60a8f,True,"config,data,results",True,True,None,2,1,False,0,...,FINISHED,2026-01-27 17:21:18.537,2026-01-28 01:08:07.691,file:///gpfs/home4/scur0063/facter-repr/mlruns...,FINISHED,2026-01-28 01:08:07.691000,2.696016,True,True,7.0
6,c332cadb09e2479c94287be997a34ea5,True,"config,data,results",True,True,None,2,1,False,0,...,FINISHED,2026-01-27 17:31:23.971,2026-01-28 00:19:21.001,file:///gpfs/home4/scur0063/facter-repr/mlruns...,FINISHED,2026-01-28 00:19:21.001000,2.729890,True,True,7.0
7,31ca9cba8c054f2588cb8881c1c7393e,True,"config,data,results",True,True,None,2,1,False,0,...,FINISHED,2026-01-27 17:32:30.261,2026-01-28 00:15:46.434,file:///gpfs/home4/scur0063/facter-repr/mlruns...,FINISHED,2026-01-28 00:15:46.434000,2.732373,True,True,7.0
8,74831232a3f14f25b62cecceeba0c801,True,"config,data,results",True,True,None,2,1,False,0,...,FINISHED,2026-01-27 17:41:33.514,2026-01-27 23:44:16.941,file:///gpfs/home4/scur0063/facter-repr/mlruns...,FINISHED,2026-01-27 23:44:16.941000,2.754242,True,True,7.0
9,d5e96806b1cc41ad9b7de840e21b3a0e,True,"config,data,results",True,True,None,2,1,False,0,...,FINISHED,2026-01-27 17:45:01.105,2026-01-27 23:41:47.788,file:///gpfs/home4/scur0063/facter-repr/mlruns...,FINISHED,2026-01-27 23:41:47.788000,2.755968,True,True,7.0


### Summary

In [67]:
summary = {
    "runs_total": int(len(df)),
    "has_results_dir": int(df["has_results_dir"].sum()),
    "has_timings": int(df["has_timings"].sum()),
    "has_stage_errors": int(df["has_stage_errors"].sum()),
    "complete_baseline": int(df["complete_baseline"].sum()),
    "complete_facter": int(df["complete_facter"].sum()),
}

print("Summary:", summary)

if "status_str" in df.columns:
    print("\nBy DB status:")
    display(df["status_str"].value_counts(dropna=False))
else:
    print("\n(DB status unavailable: zip appears to contain artifacts only.)")

print("\nTop artifact_subdirs patterns:")
display(df["artifact_subdirs"].value_counts().head(10))


Summary: {'runs_total': 64, 'has_results_dir': 47, 'has_timings': 30, 'has_stage_errors': 17, 'complete_baseline': 30, 'complete_facter': 30}

By DB status:


status_str
FINISHED    28
FAILED      17
NaN         13
RUNNING      6
Name: count, dtype: int64


Top artifact_subdirs patterns:


artifact_subdirs
config,data,results                 41
config,stage_errors                 10
config,data,results,stage_errors     6
config                               5
config,data                          1
config,data,stage_errors             1
Name: count, dtype: int64

### Build candidate set you likely want to KEEP

In [68]:
cutoff = DAYS_BACK
cand = df.copy()

unwanted = {"88667f8b7aaa4ee4aacca2a19911a32b"}
cand = cand[~cand["run_id"].isin(unwanted)]
cand = cand[cand["days_ago"] <= cutoff]

if REQUIRE_FINISHED and "status_str" in cand.columns:
    cand = cand[cand["status_str"] == "FINISHED"]

if REQUIRE_NO_STAGE_ERRORS:
    cand = cand[~cand["has_stage_errors"]]

if REQUIRE_TIMINGS:
    cand = cand[cand["has_timings"]]

if REQUIRE_FACTER_METRICS:
    cand = cand[cand["n_facter_metrics_files"] > 0]

cand = cand.sort_values(["score", "time_ref"], ascending=[False, False]).reset_index(drop=True)

print(f"Candidates kept (days_back={DAYS_BACK}):", len(cand))
display(cand.head(50))

inv_path = OUTPUT_DIR / "runs_inventory.csv"
cand_path = OUTPUT_DIR / "runs_candidates.csv"
keep_path = OUTPUT_DIR / "keep_run_ids.txt"

df.to_csv(inv_path, index=False)
cand.to_csv(cand_path, index=False)
keep_path.write_text("\n".join(cand["run_id"].tolist()), encoding="utf-8")

print("Wrote:")
print(" -", inv_path)
print(" -", cand_path)
print(" -", keep_path)


Candidates kept (days_back=5): 24


,run_id,has_artifacts,artifact_subdirs,has_results_dir,has_timings,total_seconds,n_baseline_metrics_files,n_facter_metrics_files,has_stage_errors,n_stage_errors,...,status,start_time,end_time,artifact_uri,status_str,time_ref,days_ago,complete_baseline,complete_facter,score
0,0266519be494477c8bac7b275444342c,True,"config,data,results",True,True,None,2,1,False,0,...,FINISHED,2026-01-28 15:26:41.243,2026-01-28 17:49:33.818,file:///gpfs/home4/scur0063/facter-repr/mlruns...,FINISHED,2026-01-28 17:49:33.818,2.000575,True,True,7.0
1,927cf28d77a94d95a9ac828e2fc8b730,True,"config,data,results",True,True,None,2,1,False,0,...,FINISHED,2026-01-28 12:18:28.048,2026-01-28 15:43:04.184,file:///gpfs/home4/scur0063/facter-repr/mlruns...,FINISHED,2026-01-28 15:43:04.184,2.088418,True,True,7.0
2,b11fa34edc0946f0b382275066f60a8f,True,"config,data,results",True,True,None,2,1,False,0,...,FINISHED,2026-01-27 17:21:18.537,2026-01-28 01:08:07.691,file:///gpfs/home4/scur0063/facter-repr/mlruns...,FINISHED,2026-01-28 01:08:07.691,2.696016,True,True,7.0
3,c332cadb09e2479c94287be997a34ea5,True,"config,data,results",True,True,None,2,1,False,0,...,FINISHED,2026-01-27 17:31:23.971,2026-01-28 00:19:21.001,file:///gpfs/home4/scur0063/facter-repr/mlruns...,FINISHED,2026-01-28 00:19:21.001,2.729890,True,True,7.0
4,31ca9cba8c054f2588cb8881c1c7393e,True,"config,data,results",True,True,None,2,1,False,0,...,FINISHED,2026-01-27 17:32:30.261,2026-01-28 00:15:46.434,file:///gpfs/home4/scur0063/facter-repr/mlruns...,FINISHED,2026-01-28 00:15:46.434,2.732373,True,True,7.0
5,74831232a3f14f25b62cecceeba0c801,True,"config,data,results",True,True,None,2,1,False,0,...,FINISHED,2026-01-27 17:41:33.514,2026-01-27 23:44:16.941,file:///gpfs/home4/scur0063/facter-repr/mlruns...,FINISHED,2026-01-27 23:44:16.941,2.754242,True,True,7.0
6,d5e96806b1cc41ad9b7de840e21b3a0e,True,"config,data,results",True,True,None,2,1,False,0,...,FINISHED,2026-01-27 17:45:01.105,2026-01-27 23:41:47.788,file:///gpfs/home4/scur0063/facter-repr/mlruns...,FINISHED,2026-01-27 23:41:47.788,2.755968,True,True,7.0
7,bde284c7dc9e45f78fce858c2a94ec4a,True,"config,data,results",True,True,None,2,1,False,0,...,FINISHED,2026-01-27 18:02:29.959,2026-01-27 23:03:08.733,file:///gpfs/home4/scur0063/facter-repr/mlruns...,FINISHED,2026-01-27 23:03:08.733,2.782809,True,True,7.0
8,9cee2ee3f4e94ca0aa4606a32e2ab207,True,"config,data,results",True,True,None,2,1,False,0,...,FINISHED,2026-01-27 18:02:30.203,2026-01-27 22:59:28.484,file:///gpfs/home4/scur0063/facter-repr/mlruns...,FINISHED,2026-01-27 22:59:28.484,2.785358,True,True,7.0
9,d7eb3b117b484236b8b421559efcca06,True,"config,data,results",True,True,None,2,1,False,0,...,FINISHED,2026-01-27 17:31:32.331,2026-01-27 22:35:13.527,file:///gpfs/home4/scur0063/facter-repr/mlruns...,FINISHED,2026-01-27 22:35:13.527,2.802198,True,True,7.0


Wrote:
 - C:\Users\daimy\Uni\MasterAI\FACT\facter-repr\scripts\mlruns\output\runs_inventory.csv
 - C:\Users\daimy\Uni\MasterAI\FACT\facter-repr\scripts\mlruns\output\runs_candidates.csv
 - C:\Users\daimy\Uni\MasterAI\FACT\facter-repr\scripts\mlruns\output\keep_run_ids.txt


### Copy selected runs into new folder

In [69]:
COPY_PRUNED = True
PRUNED_DIR = OUTPUT_DIR / "pruned_mlruns"
PRUNED_DIR.mkdir(exist_ok=True)

if COPY_PRUNED:
    import shutil
    kept = set(cand["run_id"].tolist())
    copied = 0
    # artifact-only layout: mlruns/<run_id>/...
    for run_id in kept:
        src = mlruns_dir / run_id
        if src.exists():
            dst = PRUNED_DIR / run_id
            if dst.exists():
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
            copied += 1
    print(f"Copied {copied} runs to:", PRUNED_DIR)
else:
    print("COPY_PRUNED is False (no files copied). Set COPY_PRUNED=True if you want a pruned folder to zip.")


Copied 24 runs to: C:\Users\daimy\Uni\MasterAI\FACT\facter-repr\scripts\mlruns\output\pruned_mlruns
